# MeetMind: BERT-base GPU training
Run manually in Colab/Kaggle or a Jupyter GPU environment. Enable a GPU first. This notebook downloads public AMI annotations and public BERT weights; it does not upload private meeting recordings or invoke a hosted inference API. Training occurs on the chosen runtime. Save the final ZIP before an ephemeral runtime ends.
The two learning-rate candidates are selected using validation macro F1. Test evaluation happens only after selection. Results are dialogue-act metrics, not confirmed commitment/owner/deadline accuracy. See backend/BERT.md for attribution and limitations.

In [ ]:
import sys, subprocess, pathlib, os, json
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers>=4.48,<5', 'scikit-learn>=1.6,<2', 'safetensors'])
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime and use its CUDA-enabled PyTorch installation.'
print(torch.__version__, torch.cuda.get_device_name(0))
repo = pathlib.Path.cwd() / 'Meet_Mind'
if not repo.exists():
    subprocess.check_call(['git', 'clone', '--branch', 'feat/bert-dialogue-classifier', 'https://github.com/AhmadHassanSE/Meet_Mind.git', str(repo)])
backend = repo / 'meetmind' / 'backend'
os.chdir(backend)
print('Source revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
def run_module(module, *args):
    subprocess.check_call([sys.executable, '-u', '-m', module, *map(str, args)])
run_module('scripts.prepare_ami')
manifest = json.loads(pathlib.Path('data/ami-da-v1/manifest.json').read_text())
print({name: entry['rows'] for name, entry in manifest['splits'].items()})


In [ ]:
template = json.loads(pathlib.Path('configs/bert-base.json').read_text())
candidate_runs = []
for learning_rate, name in [(2e-5, 'bert-base-gpu-lr2e5'), (3e-5, 'bert-base-gpu-lr3e5')]:
    config = dict(template, learning_rate=learning_rate, output=f'runs/{name}', stage='full-gpu-candidate')
    path = pathlib.Path('configs') / f'{name}.json'
    path.write_text(json.dumps(config, indent=2))
    run_module('scripts.train_bert', '--config', path)
    candidate_runs.append(config['output'])


In [ ]:
run_module('scripts.select_bert', '--runs', *candidate_runs, '--output', 'models/dialogue-bert-gpu')
selection = json.loads(pathlib.Path('models/dialogue-bert-gpu/selection.json').read_text())
selected_run = pathlib.Path(selection['selected_run'])
# Freeze selection here. Never change candidates using the following test scores.
for split in ['validation', 'test']:
    run_module('scripts.evaluate_bert', '--checkpoint', selected_run / 'best', '--split', split, '--output', selected_run / f'{split}.json')
run_module('scripts.summarize_bert', '--run', selected_run, '--output', 'reports/bert-gpu-selected')


In [ ]:
import shutil
bundle = pathlib.Path('export/meetmind-bert-gpu')
bundle.mkdir(parents=True, exist_ok=False)
shutil.copytree('models/dialogue-bert-gpu', bundle / 'model')
shutil.copytree('reports/bert-gpu-selected', bundle / 'evidence')
archive = shutil.make_archive(str(bundle), 'zip', root_dir=bundle)
print('Download this file:', pathlib.Path(archive).resolve())
# On your laptop, extract model/ to a new model directory and set BERT_MODEL_PATH.
# Keep selection.json, the weights hash and evidence with the checkpoint.
